# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates step-by-step loading, exploration, and processing of the [FAIR²](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
We use the Croissant schema provided at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Install `mlcroissant` if not already installed
!pip install mlcroissant

## 1. Data Loading
Load the Croissant dataset, inspect its metadata, and print core info using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as object attributes
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Version: {dataset.metadata.version}")


## 2. Data Overview
Let's review the available record sets, their fields, and all relevant `@id`s from the Croissant metadata.

Record sets, fields, and columns are the natural units of tabular data in Croissant, each referenced by their `@id`.

In [ ]:
# Extract record set, field, and column info by their @id
record_sets = dataset.metadata.recordSet
if not isinstance(record_sets, list):
    record_sets = [record_sets]

print("Available Record Sets (@id):\n---------------------------")
for rs in record_sets:
    print(f"- @id: {rs['@id']}  | name: {rs.get('name','<unnamed>')}")
    if 'field' in rs:
        print("  Fields:")
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        for fld in fields:
            print(f"    - @id: {fld['@id']}, name: {fld.get('name','')} (dataType: {fld.get('dataType','')})")
    if 'column' in rs:
        print("  Columns:")
        cols = rs['column'] if isinstance(rs['column'], list) else [rs['column']]
        for col in cols:
            print(f"    - @id: {col['@id']}, name: {col.get('name','')}")
    print()

## 3. Data Extraction
For our analysis, we will extract the records from one or more record sets. 
All data access will be using the record set and field/column `@id`s, following Croissant standard.

Let's load each record set as a Pandas DataFrame.

In [ ]:
# Build a list of record set @id strings
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Load all records from this record set
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nLoaded DataFrame for Record Set @id: {record_set_id}")
    print(f"Columns: {df.columns.tolist()}")
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Let's filter, normalize, and analyze one of the numeric columns in our main record set. All operations use field `@id` references for safety and reproducibility.

Here, `cr:coverage` (example numeric field) and `cr:description` (as possible grouping) are used. Update these as appropriate for the available fields in your dataset.

In [ ]:
# Example: Select first main record set (replace with actual primary set @id if needed)
main_rs_id = record_set_ids[0]
df_main = dataframes[main_rs_id]

# We'll look for a numeric field. Let's find one by checking data types for each column
print("\nSearching for numeric columns (float/int) available:")
numeric_fields = []
for col in df_main.columns:
    if pd.api.types.is_numeric_dtype(df_main[col]):
        print(f"- {col}")
        numeric_fields.append(col)

# If at least one numeric field exists, pick one
if numeric_fields:
    numeric_field_id = numeric_fields[0]  # Using the first as example
    print(f"\nUsing numeric field (by @id): {numeric_field_id}")
else:
    print("No numeric field found! Please check the data and update the analysis.")

# Define threshold and perform filtering (on the example numeric field)
if numeric_fields:
    threshold = df_main[numeric_field_id].mean()
    filtered_df = df_main[df_main[numeric_field_id] > threshold].copy()
    print(f"\nRecords with {numeric_field_id} > {threshold:.3f}:")
    print(filtered_df[[numeric_field_id]].head())

    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized field:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a categorical field if available
    # Exclude the numeric field itself
    available_group_fields = [col for col in df_main.columns if col != numeric_field_id and not pd.api.types.is_numeric_dtype(df_main[col])]
    if available_group_fields:
        group_field_id = available_group_fields[0]
        print(f"\nGrouping by field (by @id): {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        print(grouped_df.head())
    else:
        print("\nNo suitable categorical group field found.")


## 5. Visualization
Let's plot the distribution of the numeric field and, if applicable, the mean values by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_fields:
    # Plot histogram
    plt.figure(figsize=(8,4))
    sns.histplot(df_main[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If we did groupby above
    if 'grouped_df' in locals():
        plt.figure(figsize=(10,4))
        grouped_df.plot(kind='bar', legend=False)
        plt.title(f"Mean {numeric_field_id} by group: {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load a Croissant-conformant dataset, inspect its metadata, and perform basic record set exploration, numeric filtering, normalization, and visualization—all with **entity `@id` references** for reproducibility.

This approach can be further extended to deeper analysis, multi-set integration, and downstream machine learning workflows.

Explore the [mlcroissant documentation](https://mlcommons.github.io/croissant/python/) for advanced features.